# 02 — Lire un portefeuille à partir des transactions V2

Cette vue reconstitue des flux signés et un inventaire. Une valeur marquée au dernier prix est une **estimation**, pas un PnL réalisé.

In [ ]:
import polars as pl

from poly_data.analysis.positions import expand_to_positions
from poly_data.io.parquet_store import ParquetStore
from poly_data.notebooks import resolve_notebook_context, source_inventory

ctx = resolve_notebook_context()
store = ParquetStore(ctx.root)
SOURCES = ['trades', 'markets_current', 'market_outcomes']
print({'root': str(ctx.root), 'mode': ctx.mode, 'revision': ctx.revision})
source_inventory(store, SOURCES)

## Flux de trésorerie et inventaire

Chaque transaction est développée côté maker et taker. Un achat dépense du cash et ajoute des jetons ; une vente fait l'inverse. Nous choisissons ici un portefeuille déterministe de la fixture, puis agrégeons ses positions.

In [ ]:
positions = expand_to_positions(store.scan('trades'), player_side='both').collect()
wallet = positions.select('player').unique().sort('player').item(0, 0)
wallet_positions = (
    positions.filter(pl.col('player') == wallet)
    .group_by(['market_id', 'token_side'])
    .agg([
        pl.col('signed_tokens').sum().alias('net_tokens'),
        pl.col('signed_usd').sum().alias('net_cash_flow'),
        pl.col('timestamp').max().alias('last_trade_at'),
    ])
)
print({'wallet': wallet, 'positions': wallet_positions.height})
wallet_positions.head(10)

In [ ]:
last_prices = (
    store.scan('trades').sort('timestamp')
    .group_by(['market_id', 'nonusdc_side']).last()
    .select(['market_id', pl.col('nonusdc_side').alias('token_side'), pl.col('price').alias('mark_price')])
    .collect()
)
marked = (
    wallet_positions.join(last_prices, on=['market_id', 'token_side'], how='left')
    .with_columns((pl.col('net_tokens') * pl.col('mark_price') + pl.col('net_cash_flow')).alias('estimated_value'))
    .sort('estimated_value')
)
marked.head(10)

## Provenance d'une résolution

Les lignes ci-dessous joignent l'inventaire au résultat officiel lorsque le marché est clos. L'absence de résultat signifie que le portefeuille reste marqué, non réglé.

In [ ]:
official = store.scan('market_outcomes').select(['market_id', 'winner_token', 'resolved_at']).collect()
markets = store.scan('markets_current').select(['id', 'question']).collect()
(
    marked.join(official, on='market_id', how='left')
    .join(markets, left_on='market_id', right_on='id')
    .select(['market_id', 'question', 'token_side', 'net_tokens', 'mark_price', 'estimated_value', 'winner_token', 'resolved_at'])
    .head(15)
)